In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../raw data/Customer_Churn_Dataset.csv')

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,numAdminTickets,numTechTickets,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.5,0,0,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,0,0,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,3,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,0,0,Yes


In [2]:
# Data Preprocessing 
# Ép kiểu cột TotalCharges sang dạng số (float), những dòng lỗi khoảng trắng sẽ thành NaN (rỗng)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Điền các giá trị rỗng này bằng số 0 (vì họ chưa thanh toán tháng nào)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Bỏ cột customerID ra khỏi tập huấn luyện vì nó không mang ý nghĩa dự đoán
# (Nhưng ta sẽ giữ lại ID ở một biến khác để lát nữa ghép kết quả)
customer_ids = df['customerID']
df_model = df.drop(['customerID'], axis=1)

print("Đã xử lý xong TotalCharges và loại bỏ customerID!")

Đã xử lý xong TotalCharges và loại bỏ customerID!


In [3]:
# Data Encoding
from sklearn.preprocessing import LabelEncoder

# Chuyển đổi các biến phân loại (Categorical variables) thành số
le = LabelEncoder()

# Lặp qua các cột có định dạng là 'object' (chữ) để mã hóa
for col in df_model.select_dtypes(include=['object']).columns:
    df_model[col] = le.fit_transform(df_model[col])

df_model.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_14884\1903962213.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_model.select_dtypes(include=['object']).columns:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,numAdminTickets,numTechTickets,Churn
0,0,0,1,0,1,0,1,0,0,2,...,0,0,0,1,2,29.85,29.85,0,0,0
1,1,0,0,0,34,1,0,0,2,0,...,0,0,1,0,3,56.95,1889.50,0,0,0
2,1,0,0,0,2,1,0,0,2,2,...,0,0,0,1,3,53.85,108.15,0,0,1
3,1,0,0,0,45,0,1,0,2,0,...,0,0,1,0,0,42.30,1840.75,0,3,0
4,0,0,0,0,2,1,0,1,0,0,...,0,0,0,1,2,70.70,151.65,0,0,1


In [4]:
# Model Training
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Chia biến độc lập (X) và biến mục tiêu cần dự đoán (y = Churn)
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

# Tách dữ liệu: 80% để train, 20% để test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Khởi tạo và huấn luyện mô hình Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Đánh giá độ chính xác của mô hình trên tập Test
y_pred = rf_model.predict(X_test)
print("Độ chính xác của mô hình (Accuracy):", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

Độ chính xác của mô hình (Accuracy): 84.95 %


In [5]:
# Predict Probabilioty of Churn and Prepare for Power BI
# Yêu cầu mô hình dự đoán XÁC SUẤT rời bỏ trên toàn bộ danh sách khách hàng ban đầu
churn_probabilities = rf_model.predict_proba(X)[:, 1] # Lấy cột xác suất của trường hợp "Yes"

# Tạo một DataFrame mới gộp ID khách hàng, thông tin quan trọng và Xác suất rời bỏ
df_final = df[['customerID', 'Contract', 'InternetService', 'MonthlyCharges', 'numTechTickets', 'Churn']].copy()
df_final['Churn_Probability_%'] = np.round(churn_probabilities * 100, 2)

# Lọc ra danh sách những khách hàng ĐANG SỬ DỤNG (Churn = No) nhưng có RỦI RO CAO (>60%)
high_risk_customers = df_final[(df_final['Churn'] == 'No') & (df_final['Churn_Probability_%'] > 60)]

print(f"Phát hiện {len(high_risk_customers)} khách hàng có rủi ro rời bỏ cao!")

# Lưu file kết quả này về máy để Giai đoạn 3 đưa lên Power BI vẽ Dashboard
df_final.to_csv('Churn_Predictions_For_PowerBI.csv', index=False)
print("Đã lưu file thành công!")

Phát hiện 56 khách hàng có rủi ro rời bỏ cao!
Đã lưu file thành công!
